In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw"

print("Dataset directory:", os.path.abspath(DATA_DIR))
print("Files:")
for file in os.listdir(DATA_DIR):
    print(" -", file)

Dataset directory: D:\projects\AI-IDS\data\raw
Files:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


In [2]:
# Step 3: Inspect the first CSV file

import os
import pandas as pd

monday_file = os.path.join(
    DATA_DIR,
    "Monday-WorkingHours.pcap_ISCX.csv"
)

df_sample = pd.read_csv(
    monday_file,
    nrows=5
)

print("Shape of sample:", df_sample.shape)

print("\nColumns:")
for i, col in enumerate(df_sample.columns):
    print(i, repr(col))

Shape of sample: (5, 79)

Columns:
0 ' Destination Port'
1 ' Flow Duration'
2 ' Total Fwd Packets'
3 ' Total Backward Packets'
4 'Total Length of Fwd Packets'
5 ' Total Length of Bwd Packets'
6 ' Fwd Packet Length Max'
7 ' Fwd Packet Length Min'
8 ' Fwd Packet Length Mean'
9 ' Fwd Packet Length Std'
10 'Bwd Packet Length Max'
11 ' Bwd Packet Length Min'
12 ' Bwd Packet Length Mean'
13 ' Bwd Packet Length Std'
14 'Flow Bytes/s'
15 ' Flow Packets/s'
16 ' Flow IAT Mean'
17 ' Flow IAT Std'
18 ' Flow IAT Max'
19 ' Flow IAT Min'
20 'Fwd IAT Total'
21 ' Fwd IAT Mean'
22 ' Fwd IAT Std'
23 ' Fwd IAT Max'
24 ' Fwd IAT Min'
25 'Bwd IAT Total'
26 ' Bwd IAT Mean'
27 ' Bwd IAT Std'
28 ' Bwd IAT Max'
29 ' Bwd IAT Min'
30 'Fwd PSH Flags'
31 ' Bwd PSH Flags'
32 ' Fwd URG Flags'
33 ' Bwd URG Flags'
34 ' Fwd Header Length'
35 ' Bwd Header Length'
36 'Fwd Packets/s'
37 ' Bwd Packets/s'
38 ' Min Packet Length'
39 ' Max Packet Length'
40 ' Packet Length Mean'
41 ' Packet Length Std'
42 ' Packet Length Varia

In [3]:
print(df_sample.columns[-5:])
print("Last column:", repr(df_sample.columns[-1]))

Index(['Idle Mean', ' Idle Std', ' Idle Max', ' Idle Min', ' Label'], dtype='str')
Last column: ' Label'


In [4]:
# Step 5: Check labels in the Monday dataset

df_monday = pd.read_csv(
    monday_file,
    low_memory=False
)

# Remove leading/trailing spaces from column names
df_monday.columns = df_monday.columns.str.strip()

print("Dataset shape:", df_monday.shape)

print("\nTraffic labels:")
print(df_monday["Label"].value_counts())

Dataset shape: (529918, 79)

Traffic labels:
Label
BENIGN    529918
Name: count, dtype: int64


In [5]:
# Step 6: Check attack labels in the Wednesday dataset

wednesday_file = os.path.join(
    DATA_DIR,
    "Wednesday-workingHours.pcap_ISCX.csv"
)

df_wednesday = pd.read_csv(
    wednesday_file,
    low_memory=False
)

df_wednesday.columns = df_wednesday.columns.str.strip()

print("Dataset shape:", df_wednesday.shape)

print("\nTraffic labels:")
print(df_wednesday["Label"].value_counts())

Dataset shape: (692703, 79)

Traffic labels:
Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64


In [6]:
# Step 7: Overall label distribution across all CIC-IDS2017 files

from collections import Counter

label_counts = Counter()

for file in sorted(os.listdir(DATA_DIR)):
    if not file.lower().endswith(".csv"):
        continue

    path = os.path.join(DATA_DIR, file)

    print("Processing:", file)

    for chunk in pd.read_csv(
        path,
        chunksize=100_000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()

        counts = chunk["Label"].value_counts()

        for label, count in counts.items():
            label_counts[label] += int(count)

print("\n========== OVERALL LABEL DISTRIBUTION ==========")

for label, count in label_counts.most_common():
    print(f"{label:35} {count:,}")

print("\nTotal flows:", sum(label_counts.values()))

Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Processing: Monday-WorkingHours.pcap_ISCX.csv
Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Processing: Tuesday-WorkingHours.pcap_ISCX.csv
Processing: Wednesday-workingHours.pcap_ISCX.csv

========== OVERALL LABEL DISTRIBUTION ==========
BENIGN                              2,273,097
DoS Hulk                            231,073
PortScan                            158,930
DDoS                                128,027
DoS GoldenEye                       10,293
FTP-Patator                         7,938
SSH-Patator                         5,897
DoS slowloris                       5,796
DoS Slowhttptest                    5,499
Bot                                 1,966
Web Attack � Brute Force            1,507
Web Attack � XSS

In [7]:
# Step 8: Check missing, infinite, duplicate, and data-type issues

# Use the Wednesday dataset as an initial representative sample
df_check = df_wednesday.copy()

# Make column names consistent
df_check.columns = df_check.columns.str.strip()

print("========== BASIC INFORMATION ==========")
print("Rows:", len(df_check))
print("Columns:", len(df_check.columns))

print("\n========== MISSING VALUES ==========")
missing = df_check.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("No missing values found.")
else:
    print(missing)

print("\n========== INFINITE VALUES ==========")

numeric_columns = df_check.select_dtypes(include=np.number).columns
infinite_counts = np.isinf(df_check[numeric_columns]).sum()
infinite_counts = infinite_counts[infinite_counts > 0].sort_values(ascending=False)

if len(infinite_counts) == 0:
    print("No infinite values found.")
else:
    print(infinite_counts)

print("\n========== DUPLICATE ROWS ==========")
duplicate_count = df_check.duplicated().sum()
print("Duplicate rows:", duplicate_count)

print("\n========== DATA TYPES ==========")
print(df_check.dtypes.value_counts())

print("\n========== LABELS ==========")
print(df_check["Label"].value_counts())

========== BASIC INFORMATION ==========
Rows: 692703
Columns: 79

========== MISSING VALUES ==========
Flow Bytes/s    1008
dtype: int64

========== INFINITE VALUES ==========
Flow Packets/s    1297
Flow Bytes/s       289
dtype: int64

========== DUPLICATE ROWS ==========
Duplicate rows: 81909

========== DATA TYPES ==========
int64      54
float64    24
str         1
Name: count, dtype: int64

========== LABELS ==========
Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64


In [8]:
# Step 9: Identify columns containing non-finite values

for column in numeric_columns:
    values = df_check[column].to_numpy()

    if not np.isfinite(values).all():
        print(
            f"{column}: "
            f"{np.sum(~np.isfinite(values)):,} non-finite values"
        )

Flow Bytes/s: 1,297 non-finite values
Flow Packets/s: 1,297 non-finite values
